# Pytorch tutorials

## 0 - Quickstart

### Libs and seed

Libs

In [1]:
import torch # the main pytorch library
from torch import nn # the sub-library containing neural networks
from torch.utils.data import DataLoader # an object that generates batches of data for training/testing
from torchvision import datasets # popular datasets, architectures and common image transformations
from torchvision.transforms import ToTensor # an easy way to convert PIL images to Tensors

import numpy as np # NumPy, the python array library
import random # for generating random numbers

Set the seed

In [2]:
SEED = 0 # seed for reproducibility

random.seed(SEED) # set the seed for random numbers
np.random.seed(SEED) # set the seed for numpy arrays

### Data 

Dowload data 

In [3]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST( # FashionMNIST is a dataset of Zalando's article images
    root="data", # data is downloaded to this folder
    train=True, # this is the training data
    download=True, # download if data is not found locally
    transform=ToTensor(), # convert the image to a PyTorch tensor
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST( # FashionMNIST test data
    root="data", # data is downloaded to this folder
    train=False, # this is the test data
    download=True, # download if data is not found locally
    transform=ToTensor(), # convert the image to a PyTorch tensor
)

Create data loaders

In [4]:
batch_size = 64 # the size of input data took for one iteration

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size) # create a data loader for the training data
test_dataloader = DataLoader(test_data, batch_size=batch_size) # create a data loader for the test data

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


### The model

Define the model

In [7]:
# Get cpu, gpu or mps device for training.
device = ( # check if gpu is available
    "cuda"
    if torch.cuda.is_available() # use gpu
    else "mps" # use mps
    if torch.backends.mps.is_available() # use mps
    else "cpu" # use cpu
)
print(f"Using {device} device\n")



# Define model
class NeuralNetwork(nn.Module): # define a neural network class
    def __init__(self): # initialisation method
        super().__init__() # initialise the neural network
        self.flatten = nn.Flatten() # flatten (appiattire) the input image
        self.linear_relu_stack = nn.Sequential( # create a sequence of layers
            nn.Linear(28*28, 512), # linear layer from 28*28 to 512 (28*28 is the size of the input image, 512 is the size of the output tensor)
            nn.ReLU(),  # activation function
            nn.Linear(512, 512), # linear layer from 512 to 512 (512 is the size of the input tensor, 512 is the size of the output tensor)
            nn.ReLU(), # activation function
            nn.Linear(512, 10) # linear layer from 512 to 10 (512 is the size of the input tensor, 10 is the size of the output tensor)
        )

    def forward(self, x): # forward method
        x = self.flatten(x) # flatten the input image
        logits = self.linear_relu_stack(x) # pass the input image through the sequence of layers
        return logits # return the output tensor

# Create an instance of the model
model = NeuralNetwork().to(device) # create an instance of the neural network model and send it to the device
print(model)

Using cpu device

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


### The parmaters optimization

Define the loss function 

In [8]:
loss_fn = nn.CrossEntropyLoss() # create a loss function

Define the optimizer. 

It will move on the loss function in order to get the best parameters 

In [9]:
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3) # create an optimizer (as Stochastic Gradient Descent) and pass the model parameters and learning rate

Training method

In [10]:
def train(dataloader, model, loss_fn, optimizer): # create a training function
    size = len(dataloader.dataset) # get the size of the dataset
    model.train() # set the model to training mode
    for batch, (X, y) in enumerate(dataloader): # iterate over the data
        X, y = X.to(device), y.to(device) # send the input data to the device

        # Compute prediction error
        pred = model(X) # get the model's prediction
        loss = loss_fn(pred, y) # compute the loss (between the model's prediction and the actual value)

        # Backpropagation
        loss.backward() # backpropagate the loss
        optimizer.step() # adjust the model parameters
        optimizer.zero_grad() # reset the gradients

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X) # get the loss and the current batch
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

Testing method

In [11]:
def test(dataloader, model, loss_fn): # create a test function
    size = len(dataloader.dataset)
    num_batches = len(dataloader) # get the number of batches
    model.eval() # set the model to evaluation mode (test correspondece of model.train())
    test_loss, correct = 0, 0 # initialise the test loss and the number of correct predictions
    with torch.no_grad(): # disable gradient calculation
        for X, y in dataloader: # iterate over the data
            X, y = X.to(device), y.to(device) # send the input data to the device
            pred = model(X) # get the model's prediction
            test_loss += loss_fn(pred, y).item() # compute the loss
            correct += (pred.argmax(1) == y).type(torch.float).sum().item() # compute the number of correct predictions
    test_loss /= num_batches # compute the average loss
    correct /= size # compute the accuracy
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

### Application

Apply all

In [1]:
epochs = 5

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
    
print("Done!")

Epoch 1
-------------------------------


NameError: name 'train' is not defined

### Saving/loading models

Saving model

In [13]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


Loading model

In [21]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth"))

# Print model parameters
for name, param in model.named_parameters():
    print(f"{name}: {param}")
    

linear_relu_stack.0.weight: Parameter containing:
tensor([[ 0.0037, -0.0342, -0.0095,  ...,  0.0099, -0.0254,  0.0302],
        [-0.0181,  0.0131,  0.0319,  ...,  0.0260, -0.0037,  0.0162],
        [ 0.0179, -0.0153,  0.0317,  ..., -0.0110,  0.0231,  0.0354],
        ...,
        [-0.0050,  0.0267,  0.0250,  ...,  0.0054,  0.0286, -0.0270],
        [-0.0023, -0.0320, -0.0005,  ...,  0.0006, -0.0227, -0.0194],
        [ 0.0142, -0.0279,  0.0183,  ..., -0.0253, -0.0335,  0.0274]],
       requires_grad=True)
linear_relu_stack.0.bias: Parameter containing:
tensor([ 2.4260e-02,  2.8578e-03,  2.1304e-02, -2.6699e-02, -1.8445e-02,
         3.2771e-04, -2.1536e-02, -2.1442e-02,  9.0067e-03,  1.9024e-02,
        -1.6661e-02, -9.8398e-03,  1.8796e-02, -7.0714e-03,  3.2428e-03,
         2.0403e-02, -2.3420e-02, -8.0235e-03, -2.1479e-02, -1.2572e-02,
        -1.6480e-02, -2.9817e-02, -5.9611e-04, -2.5537e-03,  1.2895e-02,
         4.2130e-03,  1.9379e-02,  3.1163e-02, -1.9652e-02, -1.3656e-02,
   

### Example of a prediction 

Use model to make predictions 

In [22]:
classes = [ # create a list of classes
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval() # set the model to evaluation mode
x, y = test_data[0][0], test_data[0][1] # get the first test data
with torch.no_grad(): # disable gradient calculation
    x = x.to(device)
    pred = model(x) # get the model's prediction
    predicted, actual = classes[pred[0].argmax(0)], classes[y] # get the predicted and actual classes
    print(f'Predicted: "{predicted}", Actual: "{actual}"') # print the predicted and actual classes

Predicted: "Ankle boot", Actual: "Ankle boot"
